In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

In [ ]:
EFFICIENTNET_FILE = 'submission_macro_f1_robust.csv'    # Alice
RESNET_FILE = 'submission (5).csv'                # David
DENSENET_FILE = 'submission_densenet_rgb.csv'      # Andrew

In [ ]:
print(f"Reading {EFFICIENTNET_FILE}, {RESNET_FILE}, {DENSENET_FILE}...")
efficientnet_df = pd.read_csv(EFFICIENTNET_FILE)
resnet_df = pd.read_csv(RESNET_FILE)
densenet_df = pd.read_csv(DENSENET_FILE)

Reading submission_macro_f1_robust.csv, submission (5).csv, submission_densenet_rgb.csv...


In [ ]:
efficientnet_df = efficientnet_df.sort_values('id').reset_index(drop=True)
resnet_df = resnet_df.sort_values('id').reset_index(drop=True)
densenet_df = densenet_df.sort_values('id').reset_index(drop=True)

In [ ]:
assert efficientnet_df['id'].equals(resnet_df['id']), ("EfficientNet and ResNet prediction IDs do not match.")

assert efficientnet_df['id'].equals(densenet_df['id']), ("EfficientNet and DenseNet prediction IDs do not match.")

In [ ]:
final_preds = []
total_abnormal_found = 0 # Number of samples ultimately classified as abnormal.
recovered_by_ranking = 0  # Number of conflicting abnormal predictions resolved by model ranking.

## Ensemble Decision Rule

Because class 0 dominates the dataset, the individual models tend to make conservative predictions and may classify uncertain abnormal cases as normal.

The ensemble therefore uses the following task-aware rules:

1. **All models predict normal (0):** classify the sample as normal.
2. **At least one model predicts abnormal (1–4):** treat the sample as abnormal.
3. **Only one abnormal prediction:** use that model's abnormal class.
4. **Multiple models agree on an abnormal class:** use the abnormal majority.
5. **Conflicting abnormal classes:** resolve the conflict using the models' standalone leaderboard ranking:

   **EfficientNet → ResNet-18 + Attention → DenseNet-121**

The ranking is based on standalone public Macro-F1 performance.

This strategy differs from conventional majority voting because normal votes do not override an abnormal prediction solely through class frequency.

In [ ]:
for i in range(len(efficientnet_df)):
    efficientnet_pred = efficientnet_df.iloc[i]['label'] # Rank 1
    resnet_pred = resnet_df.iloc[i]['label'] # Rank 2
    densenet_pred = densenet_df.iloc[i]['label'] # Rank 3

    
    votes = [efficientnet_pred, resnet_pred, densenet_pred]

    # Ignore normal votes once at least one model detects an abnormal class.
    abnormal_votes = [v for v in votes if v != 0]

    if len(abnormal_votes) == 0:
        # All three models agree that the sample is normal.
        final_pred = 0
    else:
        total_abnormal_found += 1

        # Count votes among abnormal classes only.
        counts = Counter(abnormal_votes)
        most_common = counts.most_common() # e.g. [(1, 2), (2, 1)]

        winner, count = most_common[0]

        # A tie occurs when different abnormal classes receive
        # the same number of votes.

        if len(most_common) > 1 and most_common[0][1] == most_common[1][1]:
            
            recovered_by_ranking += 1

            cand_a = most_common[0][0]
            cand_b = most_common[1][0]

            # Resolve the tie using standalone model performance.
            if efficientnet_pred == cand_a or efficientnet_pred == cand_b:
                final_pred = efficientnet_pred 
            elif resnet_pred == cand_a or resnet_pred == cand_b:
                final_pred = resnet_pred
            else:
                final_pred = winner 
        else:
            final_pred = winner

    final_preds.append(final_pred)

In [ ]:
submission = pd.DataFrame({'id': efficientnet_df['id'], 'label': final_preds})

In [ ]:
output_file = 'submission_best3_model_ensemble_aggressive_rank.csv'
submission.to_csv(output_file, index=False)

In [ ]:
print("-" * 50)
print("Ensemble completed successfully.")

# Number of samples classified as abnormal by the strongest
# standalone model (EfficientNet).
efficientnet_abnormal = (
    efficientnet_df['label'] != 0
).sum()

additional_abnormal = (
    total_abnormal_found - efficientnet_abnormal
)

print(
    f"Total abnormal predictions from ensemble: "
    f"{total_abnormal_found}"
)

print(
    f"EfficientNet abnormal predictions: "
    f"{efficientnet_abnormal}"
)

print(
    f"Additional abnormal candidates identified by ensemble: "
    f"{additional_abnormal}"
)

print(
    f"Ties resolved using model-performance ranking: "
    f"{recovered_by_ranking}"
)

print(
    f"Ensemble predictions saved to: {output_file}"
)

--------------------------------------------------
Ensemble completed successfully.
Total abnormal predictions from ensemble: 3901
EfficientNet abnormal predictions: 3387
Additional abnormal candidates identified by ensemble: 514
Ties resolved using model-performance ranking: 43
Ensemble predictions saved to: submission_best3_model_ensemble_aggressive_rank.csv


In [ ]:
efficientnet_abnormal = (efficientnet_df['label'] != 0).sum()
resnet_abnormal = (resnet_df['label'] != 0).sum()
densenet_abnormal = (densenet_df['label'] != 0).sum()
print(f"EfficientNet: {efficientnet_abnormal}")
print(f"ResNet-18 + Attention: {resnet_abnormal}")
print(f"DenseNet-121: {densenet_abnormal}")

EfficientNet: 3387
ResNet-18 + Attention: 3146
DenseNet-121: 2955


## Ensemble Results

The ensemble classified **3,901 samples as abnormal**, compared with **3,387** abnormal predictions from the strongest standalone EfficientNet model.

This recovered **514 additional abnormal candidates**, while the model-ranking tie-break rule was triggered only **43 times**.

| Method | Public Macro-F1 |
|---|---:|
| DenseNet-121 | 0.72812 |
| ResNet-18 + Attention | 0.73530 |
| EfficientNet | 0.74714 |
| **Rule-Based Ensemble** | **0.76374** |

The ensemble improved public Macro-F1 from **0.74714 to 0.76374** over the strongest standalone model. The result suggests that explicitly addressing the shared false-normal bias was more effective than standard majority voting for this highly imbalanced task.